In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma

C:\Users\jains\AppData\Local\Temp\ipykernel_14676\2573709327.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import pandas as pd
import numpy as np

In [4]:
reviews_3UP = pd.read_parquet("data/reviews_3UP.parquet")
restaurant_docs = pd.read_parquet("data/restaurant_docs.parquet")
reviews  = pd.read_parquet("data/reviews_clean.parquet")
business = pd.read_parquet("data/business_clean.parquet")
mapped   = pd.read_parquet("data/mapped.parquet")
user     = pd.read_parquet("data/user_clean.parquet")
tip      = pd.read_parquet("data/tip_clean.parquet")
checkin  = pd.read_parquet("data/checkin_clean.parquet")

In [5]:
biz_cols = business[["business_id", "name", "city", "state", "postal_code","latitude", "longitude", "stars", "review_count","attributes", "categories"]].rename(columns={"stars": "biz_stars","review_count": "biz_review_count"})

user_cols = user[["user_id", "review_count", "average_stars", "yelping_since"]].rename(columns={"review_count": "user_review_count"})

df = (reviews_3UP.merge(biz_cols,  on="business_id", how="left").merge(user_cols, on="user_id", how="left").merge(checkin[["business_id", "checkinCount", "latestCheckin"]], on="business_id", how="left"))

In [6]:
import ast

def parse_attrs(a):
    if not isinstance(a, dict): return {}
    out = {}
    for k, v in a.items():
        try: out[k] = ast.literal_eval(v) if isinstance(v, str) and v[:1] in "{u'TF" else v
        except Exception: out[k] = v
    return out

attrs = df["attributes"].map(parse_attrs)
df["price_range"]  = attrs.map(lambda d: d.get("RestaurantsPriceRange2"))   # 1–4 ($ to $$$$)
df["takeout"]      = attrs.map(lambda d: d.get("RestaurantsTakeOut") == True)
df["delivery"]     = attrs.map(lambda d: d.get("RestaurantsDelivery") == True)
df["reservations"] = attrs.map(lambda d: d.get("RestaurantsReservations") == True)
df["outdoor"]      = attrs.map(lambda d: d.get("OutdoorSeating") == True)
df["good_for_kids"]= attrs.map(lambda d: d.get("GoodForKids") == True)

In [ ]:
df

In [7]:
docs = restaurant_docs.copy()
docs["tagged_text"] = (
    docs["business_id"] + " " + docs["all_text"].str.replace(r"\s+", " ", regex=True)
)
with open("tagged_reviews.txt", "w", encoding="utf-8") as f:
    for line in docs["tagged_text"]:
        f.write(line + "\n")

In [8]:
from langchain_community.document_loaders import TextLoader
raw_documents = TextLoader("tagged_reviews.txt", encoding="utf-8").load()

In [9]:
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=1, chunk_overlap=0, separator="\n")
documents = text_splitter.split_documents(raw_documents)

Created a chunk of size 8740, which is longer than the specified 1
Created a chunk of size 305, which is longer than the specified 1
Created a chunk of size 25664, which is longer than the specified 1
Created a chunk of size 8018, which is longer than the specified 1
Created a chunk of size 28423, which is longer than the specified 1
Created a chunk of size 6602, which is longer than the specified 1
Created a chunk of size 6851, which is longer than the specified 1
Created a chunk of size 15788, which is longer than the specified 1
Created a chunk of size 25690, which is longer than the specified 1
Created a chunk of size 9484, which is longer than the specified 1
Created a chunk of size 7034, which is longer than the specified 1
Created a chunk of size 4181, which is longer than the specified 1
Created a chunk of size 49008, which is longer than the specified 1
Created a chunk of size 14287, which is longer than the specified 1
Created a chunk of size 27062, which is longer than the s

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = Chroma.from_documents(documents, embedding=embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\jains\Desktop\Palate_git\restarunt-Rec\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jains\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
def retrieve_semantic_recommendations(query: str, k: int = 5):
    results = db.similarity_search(query, k=k)
    ids = [r.page_content.split()[0] for r in results]
    return restaurant_docs[restaurant_docs["business_id"].isin(ids)]


In [15]:
("indoor, cozy place for indian")

,business_id,all_text,avg_stars,n_reviews
66,-i0nH5vp4oeoRpkVGfAQSA,Not really the town you would expect to find a...,4.434783,23
1607,INWHqjudS0vDcVKlHyz69A,"The food was quite good, with the usual assort...",4.066116,121
1869,LY6OCkHEwfZtdOoQawNkWg,What a wonderful find ! \n\nI stopped in for l...,4.387097,31
2306,QvjY9esYNRMe_5kvT6TC5g,"This place is a gem.Attractive space, great lo...",4.170213,47
4863,tm94_vqmMMeAv01L-7FJzQ,It used to be Kashmir Garden and taken over by...,5.000000,2
